# Table 1 Feature statistics per organoid class

Problems (see `../../../submission/PROBLEMS.md` for more info)

- hCOL^APP uses 500 spikes minimum rather than 5000. Is this a bug?
- The two MK-8722 files now contain a positive-magnitude `Amplitude (uV)` field. The final table used signed trough-to-post-trough-peak amplitude, recovered here as `Trough (uV) - Peak B (uV)`.
- For hCO amplitude IQR this notebook calculates 34.64 uV from the current pickle, while the submission table reports 36.64 uV. Is this just a typo in the submission?

In [3]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from manuscript_analysis.contract import DNU_SESSIONS, find_project_root
from manuscript_analysis.figures.export import save_label_variants


ROOT = find_project_root()


INPUT = ROOT / 'input'
OUTPUT = ROOT / 'output' / 'tables' / '1'
OUTPUT.mkdir(parents=True, exist_ok=True)
FIGURE_STEM = '1_feature_statistics_per_class'

MIN_SPIKES = 5_000

CONDITIONS = (
    {
        'organoid_class': 'hCO',
        'path': INPUT / 'unsorted' / 'CTRL-CO-DAY150.pkl',
        'amplitude_method': 'Amplitude (uV)',
    },
    {
        'organoid_class': 'hCO^APP',
        'path': INPUT / 'unsorted' / 'AD-CO-DAY150.pkl',
        'amplitude_method': 'Amplitude (uV)',
    },
    {
        'organoid_class': 'hCO^APP + 10 nM MK-8722',
        'path': INPUT / 'drug_test' / 'AD-CO-DAY150-MKAD2-77-10NM-30DAYS.pkl',
        'amplitude_method': 'Trough (uV) - Peak B (uV)',
    },
    {
        'organoid_class': 'hCO^APP + 100 nM MK-8722',
        'path': INPUT / 'drug_test' / 'AD-CO-DAY150-MKAD2-77-100NM-30DAYS.pkl',
        'amplitude_method': 'Trough (uV) - Peak B (uV)',
    },
    {
        'organoid_class': 'hCOL^APP',
        'path': INPUT / 'drug_test' / 'OL-AD-CO-DAY150.pkl',
        'amplitude_method': 'Amplitude (uV)',
    },
)

FEATURE_COLUMNS = {
    'peak_a': 'Peak A (uV)',
    'cisi': 'Channel ISI (ms)',
    'tpt': 'Trough to Peak (ms)',
}


In [4]:
from manuscript_analysis.data import canonical_session_name

DNU_KEYS = {canonical_session_name(name).casefold() for name in DNU_SESSIONS}


def signed_amplitude(frame: pd.DataFrame, method: str) -> pd.Series:
    if method == 'Amplitude (uV)':
        return frame['Amplitude (uV)']
    if method == 'Trough (uV) - Peak B (uV)':
        return frame['Trough (uV)'] - frame['Peak B (uV)']
    raise ValueError(f'Unknown amplitude method: {method}')


def median_and_iqr(values: pd.Series) -> tuple[float, float]:
    values = pd.to_numeric(values, errors='raise').dropna()
    return float(values.median()), float(values.quantile(.75) - values.quantile(.25))


def valid_session_count(frame: pd.DataFrame) -> int:
    counts = frame.groupby('session', sort=False).size()
    valid = counts.index.to_series().map(canonical_session_name).str.casefold().map(
        lambda key: key not in DNU_KEYS
    )
    return int(((counts >= MIN_SPIKES) & valid.to_numpy()).sum())


rows, pooled = [], {key: [] for key in ('amplitude', *FEATURE_COLUMNS)}
for condition in CONDITIONS:
    frame = pd.read_pickle(condition['path'])
    amplitude = signed_amplitude(frame, condition['amplitude_method'])
    row = {
        'organoid_class': condition['organoid_class'],
        'source_file': str(condition['path'].relative_to(ROOT)),
        'sessions': int(frame['session'].nunique()),
        'spikes': int(len(frame)),
        'valid_sessions': valid_session_count(
            frame,
        ),
    }
    row['amplitude_median_uv'], row['amplitude_iqr_uv'] = median_and_iqr(amplitude)
    pooled['amplitude'].append(amplitude)
    for key, column in FEATURE_COLUMNS.items():
        row[f'{key}_median'], row[f'{key}_iqr'] = median_and_iqr(frame[column])
        pooled[key].append(frame[column])
    rows.append(row)

summary = {
    'organoid_class': 'Summary',
    'source_file': 'pooled condition-level inputs',
    'sessions': sum(row['sessions'] for row in rows),
    'spikes': sum(row['spikes'] for row in rows),
    'valid_sessions': sum(row['valid_sessions'] for row in rows),
}
summary['amplitude_median_uv'], summary['amplitude_iqr_uv'] = median_and_iqr(
    pd.concat(pooled['amplitude'], ignore_index=True)
)
for key in FEATURE_COLUMNS:
    summary[f'{key}_median'], summary[f'{key}_iqr'] = median_and_iqr(
        pd.concat(pooled[key], ignore_index=True)
    )
rows.append(summary)

table = pd.DataFrame(rows).rename(columns={
    'peak_a_median': 'peak_a_median_uv',
    'peak_a_iqr': 'peak_a_iqr_uv',
    'cisi_median': 'cisi_median_ms',
    'cisi_iqr': 'cisi_iqr_ms',
    'tpt_median': 'tpt_median_ms',
    'tpt_iqr': 'tpt_iqr_ms',
})

output_path = OUTPUT / '1_feature_statistics_per_class.csv'
table.to_csv(output_path, index=False, float_format='%.6f')

fig, ax = plt.subplots(figsize=(18, 2.8))
ax.axis('off')
rendered_table = ax.table(
    cellText=table.round(3).astype(str).to_numpy(),
    colLabels=table.columns,
    loc='center',
    cellLoc='center',
)
rendered_table.auto_set_font_size(False)
rendered_table.set_fontsize(6)
rendered_table.scale(1, 1.35)
saved_paths = save_label_variants(fig, OUTPUT, FIGURE_STEM, bbox_inches='tight', facecolor='white')

print(f'Wrote {output_path.relative_to(ROOT)}')
for saved_path in saved_paths.values():
    print(f'Saved {saved_path.relative_to(ROOT)}')
table

Wrote output\tables\1\1_feature_statistics_per_class.csv


,organoid_class,source_file,sessions,spikes,valid_sessions,amplitude_median_uv,amplitude_iqr_uv,peak_a_median_uv,peak_a_iqr_uv,cisi_median_ms,cisi_iqr_ms,tpt_median_ms,tpt_iqr_ms
0,hCO,input\unsorted\CTRL-CO-DAY150.pkl,24,2282223,19,-58.78,34.64,8.91,15.9903,174.00,304.50,0.48,0.300
1,hCO^APP,input\unsorted\AD-CO-DAY150.pkl,33,10464522,31,-76.70,69.72,11.94,25.7170,42.10,82.55,0.38,0.277
2,hCO^APP + 10 nM MK-8722,input\drug_test\AD-CO-DAY150-MKAD2-77-10NM-30D...,12,6516509,12,-63.07,41.89,12.00,17.0080,58.35,162.05,0.43,0.255
3,hCO^APP + 100 nM MK-8722,input\drug_test\AD-CO-DAY150-MKAD2-77-100NM-30...,13,6099906,13,-67.60,53.63,14.00,26.0350,38.55,56.60,0.39,0.355
4,hCOL^APP,input\drug_test\OL-AD-CO-DAY150.pkl,7,452151,3,-65.10,32.02,8.84,14.0000,51.10,204.70,0.43,0.300
5,Summary,pooled condition-level inputs,89,25815311,78,-68.47,53.55,11.97,21.9700,46.80,115.10,0.39,0.300
